In [2]:
import os

# Prepend the folder containing cdo to PATH
os.environ["PATH"] = "/sw/spack-levante/cdo-2.2.2-4z4icb/bin:" + os.environ["PATH"]

from cdo import Cdo
cdo = Cdo()
print(cdo.version())

2.2.2


In [11]:
#!/usr/bin/env python3
import os
import glob
import re
import numpy as np
import xarray as xr

# -----------------------
# User settings
# -----------------------
DERIVED_MONTHLY_DIR = "/work/uc1275/u301827/02_MSE/full_midlatitude/derived_monthly_metpy"
SCRATCH_DIR         = "/scratch/u/u301827/full_midlatitude/tdailymax_yearly"

# If your monthly files are named like: derived_YYYYMM.nc
MONTHLY_PATTERN     = os.path.join(DERIVED_MONTHLY_DIR, "derived_*.nc")

# Which months count as JJA
JJA_MONTHS = (6, 7, 8)

# Event variable: yearly maximum of t_dailymax (per grid cell)
EVENT_VAR = "t_dailymax"

# Variable to sample in the post-window after the hottest t_dailymax
TASMAX_NAME = "tasmax"

# Variables you want sampled at the hottest t_dailymax day (edit as you like)
VARS_AT_EVENT = [
    "tasmax",
    "q_at_tasmax", "2d_at_tasmax", "t_at_tasmax", "z_at_tasmax", "sp_at_tasmax", "blh_at_tasmax",
    "mse", "mse_sat", "t_bound", "t_bound_mse", "swvl1_at_tasmax",
    "TLCL", "zLCL", "pLCL",
]

# Post-window settings: compute max tasmax after hottest t_dailymax
DAYS_AFTER_EVENT = 3
INCLUDE_EVENT_DAY_IN_POSTWINDOW = False  # False => offsets [1..DAYS_AFTER], True => [0..DAYS_AFTER]

# Float32 for compactness (leave ints / time vars alone)
CAST_FLOAT32 = True


# -----------------------
# Helpers: parse year+month from filename
# -----------------------
def parse_yyyymm_from_filename(path):
    """
    Accepts filenames containing YYYYMM, e.g. derived_197906.nc
    Returns (year, month) as ints.
    """
    base = os.path.basename(path)
    m = re.search(r"(\d{4})(\d{2})", base)
    if not m:
        raise ValueError(f"Could not parse YYYYMM from: {path}")
    y = int(m.group(1))
    mo = int(m.group(2))
    return y, mo


def group_jja_files_by_year(monthly_files):
    years = {}
    for f in monthly_files:
        y, mo = parse_yyyymm_from_filename(f)
        if mo in JJA_MONTHS:
            years.setdefault(y, []).append(f)

    # ensure each year is sorted by month
    for y in years:
        years[y] = sorted(years[y], key=lambda p: parse_yyyymm_from_filename(p)[1])
    return years


# -----------------------
# Step 1: merge JJA months into yearly JJA file (CDO)
# -----------------------
def cdo_merge_jja_for_year(year, files_jja, out_dir):
    """
    Creates: out_dir/jja_merged_<year>.nc
    """
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"jja_merged_{year}.nc")

    if os.path.exists(out_path):
        print(f"[SKIP merge] {year} -> {out_path}")
        return out_path

    cdo = Cdo()
    print(f"[MERGE] {year}: {len(files_jja)} files -> {out_path}")
    cdo.mergetime(input=" ".join(files_jja), output=out_path, options="-O")
    return out_path


def max_tasmax_in_days_after_event(
    ds,
    tasmax_name=TASMAX_NAME,
    event_idx=None,
    days_after=3,
    include_event_day=False,
):
    """
    Robust version: handles grid cells where the entire post-window is invalid (all NaN)
    without crashing (returns NaN for max, and NaN/NaT for time there).
    """
    if "time" not in ds.dims:
        raise ValueError("Dataset has no 'time' dimension.")
    if tasmax_name not in ds:
        raise KeyError(f"'{tasmax_name}' not in dataset vars: {list(ds.data_vars)}")
    if "time" not in ds[tasmax_name].dims:
        raise ValueError(f"'{tasmax_name}' has no 'time' dimension; dims are {ds[tasmax_name].dims}")
    if event_idx is None:
        raise ValueError("event_idx must be provided (argmax indices of the EVENT variable).")

    start = 0 if include_event_day else 1
    lags = list(range(start, days_after + 1))

    ntime = ds.sizes["time"]

    tas_lag_list = []
    time_lag_list = []

    for lag in lags:
        idx_lag = event_idx + lag

        # avoid going past last time index
        valid = (idx_lag >= 0) & (idx_lag < ntime)
        idx_safe = xr.where(valid, idx_lag, 0)

        tas_lag = ds[tasmax_name].isel(time=idx_safe).where(valid)
        time_lag = ds["time"].isel(time=idx_safe).where(valid)

        tas_lag_list.append(tas_lag)
        time_lag_list.append(time_lag)

    tas_stack = xr.concat(tas_lag_list, dim="lag")
    time_stack = xr.concat(time_lag_list, dim="lag")

    # max over lag (NaN if all invalid)
    tas_post_max = tas_stack.max("lag", skipna=True)

    # SAFE argmax:
    #  - if all NaN along lag, argmax would crash -> detect those cells
    all_nan = tas_stack.isnull().all("lag")
    tas_for_argmax = tas_stack.fillna(-np.inf)
    lag_of_max = tas_for_argmax.argmax("lag")

    tas_post_max_time = time_stack.isel(lag=lag_of_max).where(~all_nan)

    return tas_post_max, tas_post_max_time


# -----------------------
# Step 2: compute hottest t_dailymax day per grid cell and sample vars at that day
# -----------------------
def reduce_to_event_day(
    ds,
    event_var=EVENT_VAR,
    vars_at_event=VARS_AT_EVENT,
    tasmax_name=TASMAX_NAME,
    days_after_event=DAYS_AFTER_EVENT,
    include_event_day_in_postwindow=INCLUDE_EVENT_DAY_IN_POSTWINDOW,
):
    """
    ds must have a 'time' dimension and event_var present.

    Returns a dataset with:
      - event_max (max of event_var; i.e., yearly max of t_dailymax over JJA)
      - event_time_* variables storing which day was selected
      - vars_at_event reduced to the gridcell-specific argmax time of event_var
      - NEW: tasmax_postmax_<Nd> = max tasmax in the Nd days AFTER the event day
      - NEW: date of that postmax
      - NO 'time' dimension in the result
    """
    if "time" not in ds.dims:
        raise ValueError("Dataset has no 'time' dimension.")
    if event_var not in ds:
        raise KeyError(f"'{event_var}' not in dataset vars: {list(ds.data_vars)}")

    ev = ds[event_var]

    spatial_dims = [d for d in ev.dims if d != "time"]
    if not spatial_dims:
        raise ValueError(f"{event_var} dims are {ev.dims}, expected time + spatial dims")

    # index/time/value of hottest t_dailymax per grid cell
    event_idx = ev.argmax("time")
    event_val = ev.max("time")
    event_time = ds["time"].isel(time=event_idx)

    # NEW: max tasmax after hottest t_dailymax
    tas_postmax = None
    tas_postmax_time = None
    if tasmax_name in ds:
        tas_postmax, tas_postmax_time = max_tasmax_in_days_after_event(
            ds,
            tasmax_name=tasmax_name,
            event_idx=event_idx,
            days_after=days_after_event,
            include_event_day=include_event_day_in_postwindow,
        )
        if CAST_FLOAT32 and np.issubdtype(tas_postmax.dtype, np.floating):
            tas_postmax = tas_postmax.astype("float32")

    # Build time helper vars (datetime64 preferred; fallback to strings)
    if np.issubdtype(event_time.dtype, np.datetime64):
        event_time_ns = event_time.astype("datetime64[s]").astype("int64")
        event_time_unix = xr.DataArray(
            event_time_ns.data,
            coords=event_time.coords,
            dims=event_time.dims,
            name="event_time_unix",
            attrs={"long_name": f"{event_var} max day timestamp", "units": "seconds since 1970-01-01 00:00:00"},
        )
        event_yyyymmdd = event_time.dt.strftime("%Y%m%d").astype("int32").rename("event_yyyymmdd")
        event_yyyymmdd.attrs.update({"long_name": f"{event_var} max day as YYYYMMDD", "units": "1"})
        event_doy = event_time.dt.dayofyear.astype("int16").rename("event_doy")
        event_doy.attrs.update({"long_name": f"Day-of-year of {event_var} max day", "units": "day"})
        event_time_str = None
    else:
        tstr = event_time.astype(str)
        event_time_str = tstr.rename("event_time_str")
        event_time_str.attrs.update({"long_name": f"{event_var} max day timestamp (string)", "units": "1"})
        ymd = xr.apply_ufunc(lambda s: int(s[:4] + s[5:7] + s[8:10]), tstr, vectorize=True)
        event_yyyymmdd = ymd.astype("int32").rename("event_yyyymmdd")
        event_yyyymmdd.attrs.update({"long_name": f"{event_var} max day as YYYYMMDD", "units": "1"})
        event_time_unix = None
        event_doy = None

    # Output dataset
    out = xr.Dataset()

    out["event_max"] = event_val.astype("float32" if CAST_FLOAT32 and np.issubdtype(event_val.dtype, np.floating) else event_val.dtype)
    out["event_max"].attrs.update({
        "long_name": f"Yearly maximum of {event_var} over JJA",
        "units": ev.attrs.get("units", "")
    })

    # time vars
    if event_time_unix is not None:
        out["event_time_unix"] = event_time_unix
        out["event_doy"] = event_doy
    else:
        out["event_time_str"] = event_time_str
    out["event_yyyymmdd"] = event_yyyymmdd

    # NEW: post-event tasmax diagnostics
    if tas_postmax is not None:
        out[f"tasmax_postmax_{days_after_event}d"] = tas_postmax
        out[f"tasmax_postmax_{days_after_event}d"].attrs.update({
            "long_name": f"Maximum {tasmax_name} in the {days_after_event} days after the hottest {event_var} day"
                         + (" (including event day)" if include_event_day_in_postwindow else " (excluding event day)"),
            "units": ds[tasmax_name].attrs.get("units", "")
        })

        if np.issubdtype(tas_postmax_time.dtype, np.datetime64):
            # Mask for valid datetimes
            valid_time = ~tas_postmax_time.isnull()
        
            # Build YYYYMMDD as integers using dt parts (safe), then apply mask
            yyyymmdd = (
                tas_postmax_time.dt.year * 10000
                + tas_postmax_time.dt.month * 100
                + tas_postmax_time.dt.day
            )
        
            yyyymmdd = xr.where(valid_time, yyyymmdd.astype("int32"), np.int32(-1))
        
            out[f"tasmax_postmax_{days_after_event}d_yyyymmdd"] = yyyymmdd
            out[f"tasmax_postmax_{days_after_event}d_yyyymmdd"].attrs.update({
                "long_name": f"Date of max {tasmax_name} in post-window (YYYYMMDD, -1 if missing)",
                "units": "1",
                "missing_value": -1,
            })
        else:
            out[f"tasmax_postmax_{days_after_event}d_time_str"] = tas_postmax_time.astype(str)
            out[f"tasmax_postmax_{days_after_event}d_time_str"].attrs.update({
                "long_name": f"Time of max {tasmax_name} in post-window (string)",
                "units": "1"
            })



    # Sample requested variables at the EVENT time (grid-cell-specific isel)
    for v in vars_at_event:
        if v not in ds:
            continue
        da = ds[v]
        if "time" not in da.dims:
            out[v] = da
            continue
        sampled = da.isel(time=event_idx)
        if CAST_FLOAT32 and np.issubdtype(sampled.dtype, np.floating):
            sampled = sampled.astype("float32")
        out[v] = sampled

    out.attrs.update({
        "description": f"Variables sampled at the grid-cell-specific hottest {event_var} day (JJA only), plus post-window tasmax diagnostics.",
        "event_var": str(event_var),
        "tasmax_source": str(tasmax_name),
        "postwindow_days_after": np.int32(days_after_event),
        "postwindow_includes_event_day": np.int8(1 if include_event_day_in_postwindow else 0),
    })

    return out


def process_year_to_event_file(year, merged_year_file, out_dir):
    """
    Reads merged JJA file for a year, writes: out_dir/event_at_tdailymax_<year>.nc
    """
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"event_at_tdailymax_{year}.nc")
    if os.path.exists(out_path):
        print(f"[SKIP event] {year} -> {out_path}")
        return out_path

    print(f"[EVNT] {year}: {merged_year_file}")
    ds = xr.open_dataset(merged_year_file)

    out = reduce_to_event_day(ds)

    out.to_netcdf(out_path)

    ds.close()
    out.close()

    print(f"[DONE] {year} -> {out_path}")
    return out_path


# -----------------------
# Main driver
# -----------------------
def main():
    monthly_files = sorted(glob.glob(MONTHLY_PATTERN))
    if not monthly_files:
        raise FileNotFoundError(f"No files found for pattern: {MONTHLY_PATTERN}")

    years = group_jja_files_by_year(monthly_files)
    if not years:
        raise RuntimeError("No JJA files found after grouping. Check naming and JJA_MONTHS.")

    print(f"[INFO] Found {len(monthly_files)} monthly files; {len(years)} years with JJA coverage.")

    merged_dir = os.path.join(SCRATCH_DIR, "merged_jja")
    event_dir  = os.path.join(SCRATCH_DIR, "tdailymax_products")

    out_files = []
    for y in sorted(years.keys()):
        files_jja = years[y]
        if len(files_jja) != 3:
            print(f"[WARN] {y}: expected 3 JJA files, found {len(files_jja)} -> {files_jja}")
        merged = cdo_merge_jja_for_year(y, files_jja, merged_dir)
        out_f  = process_year_to_event_file(y, merged, event_dir)
        out_files.append(out_f)

    print(f"[SUMMARY] wrote {len(out_files)} yearly event files into: {event_dir}")
    return out_files


if __name__ == "__main__":
    main()


[INFO] Found 255 monthly files; 85 years with JJA coverage.
[MERGE] 1940: 3 files -> /scratch/u/u301827/full_midlatitude/tdailymax_yearly/merged_jja/jja_merged_1940.nc
[EVNT] 1940: /scratch/u/u301827/full_midlatitude/tdailymax_yearly/merged_jja/jja_merged_1940.nc
[DONE] 1940 -> /scratch/u/u301827/full_midlatitude/tdailymax_yearly/tdailymax_products/event_at_tdailymax_1940.nc
[MERGE] 1941: 3 files -> /scratch/u/u301827/full_midlatitude/tdailymax_yearly/merged_jja/jja_merged_1941.nc
[EVNT] 1941: /scratch/u/u301827/full_midlatitude/tdailymax_yearly/merged_jja/jja_merged_1941.nc
[DONE] 1941 -> /scratch/u/u301827/full_midlatitude/tdailymax_yearly/tdailymax_products/event_at_tdailymax_1941.nc
[MERGE] 1942: 3 files -> /scratch/u/u301827/full_midlatitude/tdailymax_yearly/merged_jja/jja_merged_1942.nc
[EVNT] 1942: /scratch/u/u301827/full_midlatitude/tdailymax_yearly/merged_jja/jja_merged_1942.nc
[DONE] 1942 -> /scratch/u/u301827/full_midlatitude/tdailymax_yearly/tdailymax_products/event_at_tdai

In [12]:
import os
import re
import glob
import numpy as np
import xarray as xr

# NEW paths / filenames for the t_dailymax-based products
in_dir  = "/scratch/u/u301827/full_midlatitude/tdailymax_yearly/tdailymax_products/"
out_dir = "/work/uc1275/u301827/02_MSE/full_midlatitude/TDAILYMAX"
os.makedirs(out_dir, exist_ok=True)

out_file = os.path.join(out_dir, "event_at_tdailymax_merged_1940_2024.nc")

files = sorted(glob.glob(os.path.join(in_dir, "event_at_tdailymax_*.nc")))
print(f"Found {len(files)} files")

year_re = re.compile(r"event_at_tdailymax_(\d{4})\.nc$")


def datetime64ns_to_epoch_seconds(da: xr.DataArray) -> xr.DataArray:
    """Convert datetime64[ns] DataArray to int64 seconds since 1970-01-01."""
    sec = da.astype("datetime64[s]").astype("int64")
    out = xr.DataArray(
        sec.data,
        coords=da.coords,
        dims=da.dims,
        name=da.name,
        attrs=dict(da.attrs),
    )
    out.attrs["units"] = "seconds since 1970-01-01 00:00:00"
    out.attrs["calendar"] = "proleptic_gregorian"
    return out


datasets = []
for f in files:
    base = os.path.basename(f)
    m = year_re.match(base)
    if not m:
        raise ValueError(f"Could not parse year from filename: {base}")
    year = int(m.group(1))

    ds = xr.open_dataset(f)

    # Fix problematic datetime64 coords/vars BEFORE concat/writing
    if "time" in ds.coords and np.issubdtype(ds["time"].dtype, np.datetime64):
        ds = ds.assign_coords(time=datetime64ns_to_epoch_seconds(ds["time"]).rename("time"))

    # In the new outputs, the unix time variable is named event_time_unix (int seconds),
    # but keep this guard in case it was written as datetime somewhere.
    if "event_time_unix" in ds and np.issubdtype(ds["event_time_unix"].dtype, np.datetime64):
        ds["event_time_unix"] = datetime64ns_to_epoch_seconds(ds["event_time_unix"])

    # If you chose to also store the post-window time as a datetime64 variable, convert it too
    for v in list(ds.variables):
        if v.endswith("_time") and np.issubdtype(ds[v].dtype, np.datetime64):
            ds[v] = datetime64ns_to_epoch_seconds(ds[v])

    # Add merge dimension
    ds = ds.expand_dims(year=[year])
    datasets.append(ds)

    print("OK:", base)

ds_merged = xr.concat(datasets, dim="year").sortby("year")

# close file handles
for ds in datasets:
    ds.close()

# Optional compression
encoding = {v: {"zlib": True, "complevel": 4} for v in ds_merged.data_vars}

ds_merged.to_netcdf(out_file, encoding=encoding)
ds_merged.close()

print(f"\nMerged file saved to:\n{out_file}")


Found 85 files
OK: event_at_tdailymax_1940.nc
OK: event_at_tdailymax_1941.nc
OK: event_at_tdailymax_1942.nc
OK: event_at_tdailymax_1943.nc
OK: event_at_tdailymax_1944.nc
OK: event_at_tdailymax_1945.nc
OK: event_at_tdailymax_1946.nc
OK: event_at_tdailymax_1947.nc
OK: event_at_tdailymax_1948.nc
OK: event_at_tdailymax_1949.nc
OK: event_at_tdailymax_1950.nc
OK: event_at_tdailymax_1951.nc
OK: event_at_tdailymax_1952.nc
OK: event_at_tdailymax_1953.nc
OK: event_at_tdailymax_1954.nc
OK: event_at_tdailymax_1955.nc
OK: event_at_tdailymax_1956.nc
OK: event_at_tdailymax_1957.nc
OK: event_at_tdailymax_1958.nc
OK: event_at_tdailymax_1959.nc
OK: event_at_tdailymax_1960.nc
OK: event_at_tdailymax_1961.nc
OK: event_at_tdailymax_1962.nc
OK: event_at_tdailymax_1963.nc
OK: event_at_tdailymax_1964.nc
OK: event_at_tdailymax_1965.nc
OK: event_at_tdailymax_1966.nc
OK: event_at_tdailymax_1967.nc
OK: event_at_tdailymax_1968.nc
OK: event_at_tdailymax_1969.nc
OK: event_at_tdailymax_1970.nc
OK: event_at_tdailymax_1